# Feature Stores and Reuse

Welcome to the finale lesson of Section 5! This lesson crosses the bridge from traditional Data Science into the realm of **MLOps (Machine Learning Operations)**. 

When you engineer a brilliant new feature on your laptop, how do you ensure the software engineers can actually use it in the live mobile app without rewriting all your code? The answer is a Feature Store.

In the real world, engineering a feature (like calculating a customer's "30-day average spend") is only half the battle. The other half is keeping that calculation consistent across the entire company.

**The Problem: Training-Serving Skew**
1. **Training (The Lab)**: A Data Scientist uses Python and Pandas to calculate "30-day average spend" on historical data to train a model.
2. **Serving (Production)**: A Software Engineer writes Java or Go code to calculate "30-day average spend" in real-time when the customer opens the app.
3. **The Skew**: If the Engineer calculates the 30 days slightly differently than the Data Scientist did, the machine learning model will receive data it doesn't recognize and make terrible predictions. 

A **Feature Store** solves this. It is a centralized vault where engineered features are defined *once*, and then served to both the Data Scientists (for training) and the live applications (for predictions).

While industry-standard Feature Stores (like **Feast**, **Hopsworks**, or **AWS SageMaker**) require complex cloud infrastructure, we can build a simulated one in Python to understand exactly how the architecture works!

In [1]:
import pandas as pd
import numpy as np

print("✅ MLOps Environment Ready!")

✅ MLOps Environment Ready!


# 1. The Dual Nature of a Feature Store
A true Feature Store actually consists of two completely different databases working together behind the scenes:

1. **The Offline Store (For Training)**: A massive, slow database (like a Data Warehouse or Amazon S3). It contains the entire history of every feature. Data Scientists use this to grab millions of rows to train models.
2. **The Online Store (For Inference)**: A tiny, ultra-fast database (like Redis). It *only* stores the absolute latest, most recent value for a feature. The live app uses this to get split-second answers.

Let's build a Python class that simulates this dual-database architecture!

In [2]:
class MockFeatureStore:
    def __init__(self):
        # 1. The Offline Store (Stores historical data)
        self.offline_store = pd.DataFrame()
        
        # 2. The Online Store (Stores ONLY the latest data for ultra-fast lookup)
        self.online_store = {}

    def ingest_features(self, df, entity_id, timestamp_col):
        """Saves newly engineered features into both stores."""
        
        # 1. Append to Offline Store
        self.offline_store = pd.concat([self.offline_store, df]).drop_duplicates()
        
        # 2. Update Online Store (Keep only the most recent row per entity)
        # We sort by time, and keep the last (newest) record for each customer
        latest_data = df.sort_values(timestamp_col).groupby(entity_id).tail(1)
        
        for _, row in latest_data.iterrows():
            customer_id = row[entity_id]
            # Convert row to dictionary and save to our fast "Online" dictionary
            self.online_store[customer_id] = row.to_dict()
            
        print(f"✅ Ingested {len(df)} rows. Online store updated with latest values.")

    def get_historical_features(self):
        """Used by Data Scientists to train models."""
        return self.offline_store.copy()

    def get_online_features(self, entity_id):
        """Used by the Live App to make split-second predictions."""
        return self.online_store.get(entity_id, "❌ Customer not found in Online Store")

# Initialize our infrastructure
feature_store = MockFeatureStore()

# 2. Ingesting Engineered Features
Let's pretend we just finished Lesson 12. We engineered a powerful new feature called `30_day_spend`. Instead of leaving it in our personal Jupyter Notebook, we push it to the centralized Feature Store so the whole company can use it.

In [3]:
# Create some engineered historical data
engineered_data = pd.DataFrame({
    'customer_id': [101, 101, 102],
    'timestamp': pd.to_datetime(['2024-01-01', '2024-01-15', '2024-01-10']),
    '30_day_spend': [50.00, 120.50, 400.00],
    'is_vip': [0, 1, 1]
})

print("--- Data Scientist Pushes Features to the Store ---")
feature_store.ingest_features(engineered_data, entity_id='customer_id', timestamp_col='timestamp')

--- Data Scientist Pushes Features to the Store ---
✅ Ingested 3 rows. Online store updated with latest values.


# 3. Retrieving Data for Training (Offline)
A month later, a completely different Data Scientist wants to train a new model. They don't have to rewrite the Pandas code to calculate `30_day_spend` or `is_vip`. They just pull it directly from the Offline Store.

In [4]:
print("--- Data Scientist Pulls Training Data ---")
training_dataset = feature_store.get_historical_features()
display(training_dataset)

--- Data Scientist Pulls Training Data ---


,customer_id,timestamp,30_day_spend,is_vip
0,101,2024-01-01,50.0,0
1,101,2024-01-15,120.5,1
2,102,2024-01-10,400.0,1


*(Notice how the offline store has the full history of Customer 101!)*

# 4. Retrieving Data for Live Inference (Online)
The model is trained and deployed to the live website. Customer `101` clicks "Checkout." 

The website needs to predict if this customer is going to commit fraud, and it needs the answer in less than 50 milliseconds. It cannot afford to search through millions of historical rows. It hits the Online Store instead.

In [5]:
print("--- Live Website Requests Data for Customer 101 ---")
# The website asks the Feature Store for Customer 101's latest features
live_features = feature_store.get_online_features(entity_id=101)

# It gets the data instantly!
print(live_features)

--- Live Website Requests Data for Customer 101 ---
{'customer_id': 101, 'timestamp': Timestamp('2024-01-15 00:00:00'), '30_day_spend': 120.5, 'is_vip': 1}


*(Notice what happened! The online store completely ignored the old January 1st record. It instantly returned ONLY the newest January 15th record where the customer spent $120.50 and became a VIP. The machine learning model now makes a prediction using the exact same logic the Data Scientist used in training!)*

# 5. Point-in-Time Correctness (Time Travel)
One of the most complex features of a real Feature Store (like Feast) is **Point-in-Time Joins**. 

If you want to train a model to predict what a customer did on January 5th, you *cannot* use their January 15th `30_day_spend` data (that is Target Leakage!). A real feature store will mathematically "time travel" back to January 5th, perfectly stitch the state of the features as they existed on that exact day, and hand you a leak-proof training dataset. 

---

## Real-World Use Case or Analogy:
Think of a Feature Store like the **Prep Kitchen at a Michelin-Star Restaurant**:

* **The Problem (No Feature Store)**: When a customer orders a soup, the Line Cook has to run to the fridge, grab a raw onion, chop it, boil the stock, and make the soup from scratch. It takes 45 minutes. The customer leaves.
* **The Offline Store (The Prep Kitchen)**: In the morning, before the restaurant opens (Offline), the Prep Chefs chop 50 pounds of onions, make massive vats of stock, and measure out all the spices. They do the heavy lifting *once*.
* **The Online Store (The Line Cook's Station)**: During the dinner rush (Live Inference), the Line Cook has a tiny container of pre-chopped onions and pre-boiled stock right next to their stove. 
* **The Result**: When an order comes in, the Line Cook just grabs a handful of pre-processed ingredients and serves the soup in 2 minutes. The prep work (Feature Engineering) was centralized, ensuring every bowl of soup tastes identical, and is served at lightning speed.

---